Khóa 6: Introduction to APIs in Python

    Khóa 5 cho bạn cái nhìn tổng quan về việc lấy dữ liệu từ web. Khóa này đi sâu vào cách làm việc với API một cách bài bản: hiểu cấu trúc API, xử lý lỗi đúng cách, và gửi dữ liệu đi (không chỉ nhận về). Đây là kỹ năng bạn sẽ dùng liên tục khi viết pipeline ETL thật.

    Tất cả ví dụ dùng JSONPlaceholder (https://jsonplaceholder.typicode.com), một API test miễn phí, không cần key, nên bạn chạy thử được ngay.

1. REST API và Endpoint

    Hầu hết API hiện nay theo phong cách REST: mỗi loại dữ liệu có một địa chỉ riêng gọi là endpoint.
    https://jsonplaceholder.typicode.com/users        → danh sách tất cả user
    https://jsonplaceholder.typicode.com/users/1      → user có id = 1
    https://jsonplaceholder.typicode.com/posts        → danh sách bài viết
    https://jsonplaceholder.typicode.com/posts/1      → bài viết có id = 1

Cấu trúc URL

| Thành phần | Ví dụ | Vai trò |
|-----------|--------|----------|
| **Base URL** | `https://jsonplaceholder.typicode.com` | Địa chỉ gốc của API |
| **Endpoint** | `/users` | Loại tài nguyên cần lấy |
| **Query Params** | `?userId=1` | Điều kiện lọc |

Tương tự SQL: base URL giống tên database, endpoint giống tên bảng, query params giống mệnh đề WHERE.

In [1]:
import requests

BASE_URL = "https://jsonplaceholder.typicode.com"

# Lấy 1 user cụ thể
response = requests.get(f"{BASE_URL}/users/1")
user = response.json()
print(user["name"])

# Lọc bằng params: chỉ lấy bài viết của user 1
response = requests.get(f"{BASE_URL}/posts", params={"userId": 1})
posts = response.json()
print(len(posts))   # 10 bài

Leanne Graham
10


Nên tách BASE_URL ra thành biến riêng như trên. Khi API đổi địa chỉ, bạn chỉ sửa một chỗ.

2. Đọc cấu trúc JSON lồng nhau

    Dữ liệu API thật hiếm khi phẳng. Xem 1 user từ API:

In [ ]:
user = requests.get(f"{BASE_URL}/users/1").json()

In [ ]:
{
    "id": 1,
    "name": "Leanne Graham",
    "email": "Sincere@april.biz",
    "address": {                      # dict lồng trong dict
        "city": "Gwenborough",
        "geo": {"lat": "-37.3159", "lng": "81.1496"}
    },
    "company": {"name": "Romaguera-Crona"}
}

Truy cập từng tầng bằng cách nối [...]:

In [ ]:
print(user["address"]["city"])          # Gwenborough
print(user["address"]["geo"]["lat"])    # -37.3159
print(user["company"]["name"])          # Romaguera-Crona

Nhớ lại Khóa 8 track SQL: payload:address:city::STRING trong Snowflake làm đúng việc này. Cú pháp khác nhau, tư duy "đi xuống từng tầng" giống hệt nhau.

Nếu key có thể không tồn tại, dùng .get() để không bị lỗi:

In [ ]:
phone = user.get("phone_number", "Không có")   # trả về "Không có" nếu thiếu key
# user["phone_number"] sẽ văng KeyError

3. Xử lý lỗi với try / except

    Khóa 5 bạn đã biết kiểm tra status_code. Nhưng có loại lỗi xảy ra trước khi có status code: mất mạng, server không phản hồi, timeout. Lúc đó requests.get() văng exception và chương trình dừng đột ngột. Cấu trúc try/except giúp bạn bắt lỗi:

In [ ]:
try:
    response = requests.get(f"{BASE_URL}/users", timeout=5)
    response.raise_for_status()      # tự văng lỗi nếu status là 4xx hoặc 5xx
    data = response.json()
    print(f"Lấy được {len(data)} user")
except requests.exceptions.Timeout:
    print("Server phản hồi quá chậm")
except requests.exceptions.HTTPError as e:
    print(f"Lỗi HTTP: {e}")
except requests.exceptions.RequestException as e:
    print(f"Lỗi kết nối: {e}")

Ba điểm đáng nhớ:

timeout=5: nếu sau 5 giây server chưa trả lời thì bỏ cuộc. Đây mới là cách tránh "chờ đợi lâu", không phải kiểm tra status code. Luôn đặt timeout: mặc định requests chờ vô hạn, pipeline có thể treo cả đêm.
raise_for_status(): gọn hơn if status_code == 200 ... else. Mã 2xx thì bỏ qua, 4xx/5xx thì tự văng HTTPError.
Thứ tự except: đặt lỗi cụ thể lên trước, lỗi chung (RequestException) xuống cuối.

4. Gửi dữ liệu đi bằng POST

    Đến giờ bạn chỉ dùng GET (như SELECT). Khi cần tạo dữ liệu mới, dùng POST (như INSERT), kèm dữ liệu trong tham số json=:

In [ ]:
new_post = {
    "title": "Học Data Engineering",
    "body": "Hôm nay mình học API",
    "userId": 1
}

response = requests.post(f"{BASE_URL}/posts", json=new_post)
print(response.status_code)   # 201 (Created: tạo thành công)
print(response.json())        # dữ liệu server trả về, có thêm "id" mới

Lưu ý: JSONPlaceholder chỉ giả lập việc tạo, nó trả về kết quả như thật nhưng không lưu gì. Rất tiện để tập.
Tóm tắt các mã thành công:

|Mã|	Ý nghĩa|
|--|--|
|200	|OK (thường cho GET)|
|201|   	Created (tạo mới thành công, thường cho POST)|
|204	|No Content (thành công, không có nội dung trả về)|

5. Headers và API key an toàn

    Ở Khóa 5 bạn đã thấy headers chứa API key. Điểm mới ở đây là cách giấu key khỏi code:

In [ ]:
import os

api_key = os.environ.get("MY_API_KEY")    # đọc key từ biến môi trường

headers = {"Authorization": f"Bearer {api_key}"}
response = requests.get("https://api.example.com/data", headers=headers)

Bạn đặt key ở ngoài code (biến môi trường hoặc file .env). Nhờ vậy khi đẩy code lên GitHub, key không bị lộ. Đây là quy tắc bắt buộc trong công việc thật.

6. Rate limiting: đừng "tấn công" API

    Nhiều API giới hạn số request mỗi phút (vượt quá sẽ nhận mã 429). Khi lặp gọi API nhiều lần, hãy nghỉ giữa các lần:

In [ ]:
import time

for user_id in range(1, 6):
    response = requests.get(f"{BASE_URL}/users/{user_id}", timeout=5)
    print(response.json()["name"])
    time.sleep(1)    # nghỉ 1 giây giữa mỗi request

7. Ghép lại: hàm gọi API hoàn chỉnh

    Đây là cách viết chuyên nghiệp, kết hợp hàm (Khóa 3), xử lý lỗi và timeout:

In [ ]:
def lay_du_lieu(endpoint, params=None):
    """
    Gọi API GET và trả về dữ liệu JSON.

    Params:
        endpoint (str): đường dẫn, ví dụ "/users"
        params (dict): tham số lọc, mặc định None

    Returns:
        dict/list nếu thành công, None nếu thất bại
    """
    try:
        response = requests.get(f"{BASE_URL}{endpoint}", params=params, timeout=5)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Lỗi khi gọi {endpoint}: {e}")
        return None

users = lay_du_lieu("/users")
if users is not None:
    print(f"Có {len(users)} user")

🧪 Bài tập thực hành

Các bài đều chạy được với JSONPlaceholder. Bạn nên chạy thật để thấy kết quả.

Bài 1: Gọi endpoint /users/3, in ra name và tên thành phố (city) của user đó (gợi ý: city nằm trong address).

Bài 2: Gọi endpoint /posts với params={"userId": 2}. In ra số bài viết và title của bài đầu tiên.

Bài 3: Viết một đoạn code gọi /users có đủ timeout=5, raise_for_status() và try/except bắt requests.exceptions.RequestException. Nếu thành công in "Lấy được X user", nếu lỗi in "Có lỗi xảy ra".

Bài 4: Dùng POST gửi một bài viết mới tới /posts với title="Bài của tôi", body="Nội dung thử", userId=1. In ra status_code và giải thích: vì sao kết quả là 201 mà không phải 200?

Bài 5 (tổng hợp): Viết hàm lay_ten_user(user_id) nhận vào một số, gọi /users/{user_id}, trả về tên user. Nếu có lỗi (ví dụ user_id = 999 sẽ trả 404) thì trả về None thay vì làm chương trình sập. Gọi thử hàm với user_id = 1 và user_id = 999.

In [2]:
# Bài 1: Gọi endpoint /users/3, in ra name và tên thành phố (city) của user đó (gợi ý: city nằm trong address).
import requests 

BASE_URL="https://jsonplaceholder.typicode.com"
response = requests.get(f"{BASE_URL}/users/3")
user=response.json()
print(user["name"],user["address"]["city"])


Clementine Bauch McKenziehaven


In [8]:
# Bài 2: Gọi endpoint /posts với params={"userId": 2}. In ra số bài viết và title của bài đầu tiên.

import requests 

BASE_URL="https://jsonplaceholder.typicode.com" 
response= requests.get(f"{BASE_URL}/posts", params={"userId":2})
posts= response.json()
print(len(posts))
print(posts[0]["title"])

10
et ea vero quia laudantium autem


In [ ]:
# Bài 3: Viết một đoạn code gọi /users có đủ timeout=5, raise_for_status() và try/except bắt requests.exceptions.RequestException. Nếu thành công in "Lấy được X user", nếu lỗi in "Có lỗi xảy ra".
import requests
BASE_URL = "https://jsonplaceholder.typicode.com"
try:
    response= requests.get(f"{BASE_URL}/users",timeout=5)
    response.raise_for_status()
    data=response.json()
    print(f"Da lay duoc {len(data)} user")
except requests.exceptions.Timeout:
    print("Server phan hoi qua cham")
except requests.exceptions.HTTPError as e:
    print(f"Loi HTTP: {e}")
except requests.exceptions.RequestException as e:
    print(f"Loi ket noi: {e}")

Da lau duoc 10 user


In [ ]:
# Bài 4: Dùng POST gửi một bài viết mới tới /posts với title="Bài của tôi", body="Nội dung thử", userId=1. In ra status_code và giải thích: vì sao kết quả là 201 mà không phải 200?
import requests
BASE_URL = "https://jsonplaceholder.typicode.com"
new_post= {
    "title": "Bài của tôi",
    "body":"Nội dung thử",
    "userId":1
}
response = requests.post(f"{BASE_URL}/posts", json=new_post)
print(response.status_code)   # 201
print(response.json())

# 201 nghĩa là "Created": server đã tạo ra một tài nguyên mới (ở đây là bài viết mới). Mã 201 cho client biết chính xác điều gì đã xảy ra, không chỉ là "thành công" mơ hồ. Tương tự trong SQL: SELECT chỉ đọc dữ liệu, còn INSERT báo "đã thêm 1 dòng".

201
{'title': 'Bài của tôi', 'body': 'Nội dung thử', 'userId': 1, 'id': 101}


In [11]:
# Bài 5 (tổng hợp): Viết hàm lay_ten_user(user_id) nhận vào một số, gọi /users/{user_id}, trả về tên user. Nếu có lỗi (ví dụ user_id = 999 sẽ trả 404) thì trả về None thay vì làm chương trình sập. Gọi thử hàm với user_id = 1 và user_id = 999.

import requests
BASE_URL = "https://jsonplaceholder.typicode.com" 


def lay_ten_user(user_id):
    try:
        response = requests.get(f"{BASE_URL}/users/{user_id}", timeout=5)
        response.raise_for_status()
        return response.json()["name"]
    except requests.exceptions.RequestException as e:
        print(f"Loi khi goi: {e}")
        return None

print(lay_ten_user(1))     # Leanne Graham
print(lay_ten_user(999))   # in thông báo lỗi 404, rồi trả về None





Leanne Graham
Loi khi goi: 404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/users/999
None
